# Clean-only one-class detector on the shared split (Roua)

CPU runtime is enough (about 1 s of feature extraction per adapter).
Uses `shared_protocol.detectors.oneclass`: Ledoit-Wolf Mahalanobis distance on 30 all-layer spectral statistics, fitted on the **benign** adapters of the shared train split only (backdoored train adapters are never opened). No hyper-parameter is tuned. Freeze, then calibrate on clean calibration adapters.
Evaluation stays **locked** until every detector is frozen.

Each stage uses a fresh isolated interpreter. Supply the reviewed bootstrap and the expected SHA-256 of the reviewed source tree from the run record. Compute the identity on the reviewed tree using the bootstrap's identity command. Keep the bootstrap and dependency installation trusted. Restarting the notebook alone does not establish source identity.

Julia supplies the development bundle and stages permitted files. Evaluation requires a separately issued release after freeze. Keep final-holdout files and credentials with the custodian.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
REPO_DIR = Path("/content/repo")
EXPECTED_SOURCE = ""  # Paste the independently recorded reviewed source digest.
DEV_BUNDLE = "/content/drive/MyDrive/fyp_shared/oneclass_development.json"
ADAPTER_ROOT = "/content/adapters"
OUTPUT = Path("/content/oneclass-run")
OUTPUT.mkdir(exist_ok=True)
EVALUATION_RELEASE = None

In [ ]:
def run_stage(stage, filename, *extra):
    command = [sys.executable, "-I", "-S", "-B", str(REPO_DIR / "notebooks/oneclass_run.py"),
               stage, "--source", str(REPO_DIR), "--expected-source", EXPECTED_SOURCE,
               "--bundle", DEV_BUNDLE, "--adapters", ADAPTER_ROOT,
               "--output", str(OUTPUT / filename), *extra]
    completed = subprocess.run(command, check=True, text=True, stdout=subprocess.PIPE)
    print(completed.stdout, end="")
    return json.loads(completed.stdout.splitlines()[-1])

model_receipt = run_stage("fit", "model.json")

In [ ]:
threshold_receipt = run_stage("calibrate", "threshold.json", "--model", str(OUTPUT / "model.json"),
                              "--model-sha256", model_receipt["file_sha256"])

Retain the model and threshold receipts independently. Run evaluation only after Julia supplies the release bound to the frozen protocol and threshold. Stage its authorized adapter files in the evaluation environment.

In [ ]:
if EVALUATION_RELEASE is None:
    raise ValueError("A custodian-provided evaluation release is required")
evaluation_receipt = run_stage(
    "evaluate", "evaluation.json", "--model", str(OUTPUT / "model.json"),
    "--model-sha256", model_receipt["file_sha256"], "--threshold", str(OUTPUT / "threshold.json"),
    "--threshold-sha256", threshold_receipt["threshold_sha256"], "--release", EVALUATION_RELEASE,
)